In [1]:
import numpy as np
import re
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, PauliError
from qiskit.quantum_info import Statevector, Pauli, SparsePauliOp, StabilizerState, DensityMatrix
from scipy.optimize import minimize
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

from xgboost import XGBRegressor

import pandas as pd
import tqdm as tqdm

num_qubits = 12
num_layers = 4

ansatz = TwoLocal(
    num_qubits,
    rotation_blocks='ry',
    entanglement_blocks='cx',
    entanglement='circular',
    reps=num_layers,
    skip_final_rotation_layer=True,
    insert_barriers=True
)

from qiskit.quantum_info import Statevector, Pauli, SparsePauliOp, StabilizerState, DensityMatrix

observables = []
for i in range(num_qubits):
    for j in range(i+1, num_qubits):
        pauli_str = ['I'] * num_qubits
        pauli_str[i] = 'Z'
        pauli_str[j] = 'Z'
        observables.append(Pauli(''.join(pauli_str)))
        
for i in range(num_qubits):
    pauli_str = ['I'] * num_qubits
    pauli_str[i] = 'X'
    observables.append(Pauli(''.join(pauli_str)))

backend_noiseless = AerSimulator(method='statevector')

df = pd.read_csv('./data/sk-hamiltonians.csv')
energies = []

for i in tqdm.tqdm(range(len(df))):
    coefs = df['coefs'][i]
    params = df['params'][i]
    params_clean = re.sub(r'\s+', ' ', params.strip('[]\n '))
    params = np.fromstring(params_clean, sep=' ')

    result = eval(coefs)
    coefs = np.array(result, dtype=float)
    coefs = np.concatenate((coefs, np.array([1.]*num_qubits)))

    bound_circuit = ansatz.assign_parameters(params)

    transpiled = transpile(bound_circuit, backend_noiseless)
    transpiled.save_statevector()
    job = backend_noiseless.run(transpiled)
    result = job.result()
    statevector = result.get_statevector()

    noiseless_e = sum([np.real(statevector.expectation_value(obs)) for obs in observables]*coefs)
    energies.append(noiseless_e)

C:\Temp\ipykernel_17016\3490338517.py:24: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(
100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [00:06<00:00, 15.32it/s]


In [3]:
energies = np.array(energies)

In [19]:
df = pd.read_csv("./data/error_stats/error_stats_depolarization001.csv")
df_zne = pd.read_csv('./data/error_stats/zne_depolarization001.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 44.4 [36.7-56.4]; RMSE 0.046
Ridge Near: 8.2 [5.1-14.8]; RMSE 0.309
Ridge Clifford: 6.6 [3.1-15.7]; RMSE 0.410
XGB Near: 0.8 [0.6-0.9]; RMSE 2.434
XGB Clifford: 1.2 [0.8-2.1]; RMSE 1.945


In [20]:
df = pd.read_csv("./data/error_stats/error_stats_depolarization005.csv")
df_zne = pd.read_csv('./data/error_stats/zne_depolarization005.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 4.0 [3.5-5.1]; RMSE 1.669
Ridge Near: 8.0 [3.9-20.1]; RMSE 1.297
Ridge Clifford: 3.6 [2.1-7.8]; RMSE 2.393
XGB Near: 1.9 [1.6-2.1]; RMSE 3.547
XGB Clifford: 5.0 [3.4-10.3]; RMSE 1.767


In [21]:
df = pd.read_csv("./data/error_stats/error_stats_depolarization01.csv")
df_zne = pd.read_csv('./data/error_stats/zne_depolarization01.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 2.6 [2.1-3.2]; RMSE 5.371
Ridge Near: 3.6 [1.8-7.9]; RMSE 4.489
Ridge Clifford: 2.3 [1.3-4.8]; RMSE 7.062
XGB Near: 2.2 [2.0-2.5]; RMSE 4.278
XGB Clifford: 5.3 [3.4-12.5]; RMSE 2.511


In [16]:
df = pd.read_csv("./data/error_stats/error_stats_pauli001.csv")
df_zne = pd.read_csv('./data/error_stats/zne_pauli001.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 70.5 [43.4-148.9]; RMSE 0.032
Ridge Near: 7.9 [4.7-16.7]; RMSE 0.288
Ridge Clifford: 7.8 [4.6-20.5]; RMSE 0.272
XGB Near: 0.7 [0.5-0.9]; RMSE 2.363
XGB Clifford: 0.8 [0.6-1.9]; RMSE 2.094


In [17]:
df = pd.read_csv("./data/error_stats/error_stats_pauli005.csv")
df_zne = pd.read_csv('./data/error_stats/zne_pauli005.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 4.4 [3.8-5.7]; RMSE 1.372
Ridge Near: 7.0 [4.0-17.3]; RMSE 1.211
Ridge Clifford: 5.2 [3.2-12.2]; RMSE 1.433
XGB Near: 1.9 [1.6-2.1]; RMSE 3.091
XGB Clifford: 2.3 [1.7-4.9]; RMSE 2.824


In [18]:
df = pd.read_csv("./data/error_stats/error_stats_pauli01.csv")
df_zne = pd.read_csv('./data/error_stats/zne_pauli01.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 2.9 [2.3-3.5]; RMSE 3.231
Ridge Near: 5.0 [2.4-10.1]; RMSE 2.872
Ridge Clifford: 3.8 [2.3-9.2]; RMSE 3.206
XGB Near: 2.1 [1.9-2.5]; RMSE 4.012
XGB Clifford: 6.9 [3.6-13.7]; RMSE 1.948


In [12]:
df = pd.read_csv("./data/error_stats/error_stats_composite001.csv")
df_zne = pd.read_csv('./data/error_stats/zne_composite001.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 19.0 [15.1-26.4]; RMSE 0.173
Ridge Near: 7.8 [4.5-14.0]; RMSE 0.525
Ridge Clifford: 5.9 [2.7-13.3]; RMSE 0.727
XGB Near: 1.1 [0.9-1.3]; RMSE 2.844
XGB Clifford: 0.9 [0.7-1.4]; RMSE 3.534


In [13]:
df = pd.read_csv("./data/error_stats/error_stats_composite005.csv")
df_zne = pd.read_csv('./data/error_stats/zne_composite005.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 2.8 [2.4-3.6]; RMSE 3.230
Ridge Near: 5.5 [3.3-9.8]; RMSE 2.312
Ridge Clifford: 4.2 [2.8-9.4]; RMSE 2.747
XGB Near: 1.8 [1.7-2.1]; RMSE 4.589
XGB Clifford: 6.1 [3.4-10.6]; RMSE 2.155


In [15]:
df = pd.read_csv("./data/error_stats/error_stats_composite01.csv")
df_zne = pd.read_csv('./data/error_stats/zne_composite01.csv')
zne = df_zne['zne']
xgb_full = df['xgb_full']
xgb_near = df['xgb_near']
ridge_full = df['ridge_full']
ridge_near = df['ridge_near']
noisy = df['noisy']

noisy_error = np.abs(noisy - energies)
xgb_full_error = np.abs(xgb_full - energies)
xgb_near_error = np.abs(xgb_near - energies)
ridge_full_error = np.abs(ridge_full - energies)
ridge_near_error = np.abs(ridge_near - energies)
zne_error = np.abs(zne - energies)

xgb_full_err_sup = np.sort(noisy_error/xgb_full_error)
xgb_near_err_sup = np.sort(noisy_error/xgb_near_error)
ridge_full_err_sup = np.sort(noisy_error/ridge_full_error)
ridge_near_err_sup = np.sort(noisy_error/ridge_near_error)
zne_err_sup = np.sort(noisy_error/zne_error)

print(f'ZNE: {zne_err_sup[50]:.1f} [{zne_err_sup[25]:.1f}-{zne_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(zne_error**2)):.3f}')
print(f'Ridge Near: {ridge_near_err_sup[50]:.1f} [{ridge_near_err_sup[25]:.1f}-{ridge_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_near_error**2)):.3f}')
print(f'Ridge Clifford: {ridge_full_err_sup[50]:.1f} [{ridge_full_err_sup[25]:.1f}-{ridge_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(ridge_full_error**2)):.3f}')
print(f'XGB Near: {xgb_near_err_sup[50]:.1f} [{xgb_near_err_sup[25]:.1f}-{xgb_near_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_near_error**2)):.3f}')
print(f'XGB Clifford: {xgb_full_err_sup[50]:.1f} [{xgb_full_err_sup[25]:.1f}-{xgb_full_err_sup[75]:.1f}]; RMSE {np.sqrt(np.mean(xgb_full_error**2)):.3f}')

ZNE: 1.0 [1.0-1.0]; RMSE 10.592
Ridge Near: 3.4 [2.2-6.3]; RMSE 4.215
Ridge Clifford: 2.5 [1.9-3.7]; RMSE 5.035
XGB Near: 1.6 [1.4-1.7]; RMSE 7.041
XGB Clifford: 2.4 [1.7-3.0]; RMSE 5.332
